# Hafta 2 — Mekânsal Eşleştirme + Tahliye Skoru + Yayılma Hızı
**Sultan** — AteşKes (EERİS+)

Girdi olarak yangın noktası + kritik alan listesi (Hafta 1 çıktısı) alan, çıktı olarak `tahliye_skoru` ve yayılma hızı üreten fonksiyon seti.

In [1]:
import pandas as pd
from hesaplamalar import (
    en_yakin_kritik_alan, tahliye_skoru, egim_yonu_belirle,
    ruzgar_egime_uyumlu_mu, yayilma_hizi_belirle,
)

kritik_alanlar_df = pd.read_csv("hafta1_ornek_veri.csv")
kritik_alanlar = kritik_alanlar_df.to_dict("records")
kritik_alanlar_df

,bolge_id,isim,tip,lat,lon,yukseklik_metre,egim_derece,egim_yonu,risk_skoru,oncelik_skoru
0,mugla_01,İslamhaneleri,koy,37.029895,27.294703,40.0,11.9,kuzey,0.135,0.7
1,mugla_02,Bodrum Amerikan Hastanesi,hastane,37.039939,27.428962,13.0,1.7,kuzey,0.099,1.0
2,mugla_03,Kemer İlköğretim Okulu,okul,36.646920,29.362071,124.0,1.1,kuzey,0.097,0.9
3,mugla_04,İkinci Bahar Huzur Evi,huzurevi,36.855276,28.279101,5.0,9.1,dogu,0.125,0.9
4,mugla_05,Gürece,koy,37.039538,27.322432,160.0,10.8,kuzey,0.131,0.7
5,mugla_06,isimsiz,hastane,36.622309,29.115010,5.0,2.3,guney,0.101,1.0


## Mock yangın noktaları
**NOT:** FIRMS entegrasyonu (`/yangin-noktalari`) Esma'nın Hafta 1 işi. O hazır olana kadar bu fonksiyon setini test etmek için mock yangın noktaları kullanıyoruz. Fonksiyonun girdi imzası (`lat`, `lon`, `ruzgar_hizi`, `ruzgar_yonu_derece` içeren kayıt listesi) FIRMS+Open-Meteo birleşiminden gelecek gerçek veriyle aynı olacağı için entegrasyonda sadece veri kaynağı değişecek, fonksiyon değişmeyecek.

In [2]:
mock_yangin_noktalari = [
    {"yangin_id": "yangin_01", "lat": 37.03, "lon": 27.30, "ruzgar_hizi": 25, "ruzgar_yonu_derece": 0},
    {"yangin_id": "yangin_02", "lat": 36.85, "lon": 28.28, "ruzgar_hizi": 15, "ruzgar_yonu_derece": 180},
]

## Hafta 2 teslimi: yangin_kritik_alan_eslestir()
Haversine ile en yakin kritik alani bulur, `tahliye_skoru = risk x oncelik`
hesaplar. Yayilma hizi icin artik GERCEK egim yonunu kullaniyoruz: Hafta 1'de
her kritik alan icin `egim_yonu_belirle()` ile hesaplanip CSV'ye kaydedilen
`egim_yonu` sutunu + rüzgarin geldigi yon (`ruzgar_yonu_derece`) birlikte
`ruzgar_egime_uyumlu_mu()`'ya veriliyor. Onceki surumde bu iliski
`ruzgar_yonu_derece < 90` gibi gecici bir kestirimle yapiliyordu; artik
gercek 5 nokta yukseklik verisinden turetilen yon kullaniliyor.

In [3]:
def yangin_kritik_alan_eslestir(yangin_noktalari, kritik_alanlar):
    sonuclar = []
    for yangin in yangin_noktalari:
        alan, mesafe = en_yakin_kritik_alan(yangin["lat"], yangin["lon"], kritik_alanlar)
        if alan is None:
            continue
        tahliye = tahliye_skoru(alan["risk_skoru"], alan["oncelik_skoru"])

        # Gercek eğim yönü (Hafta 1'de egim_yonu_belirle() ile hesaplanip
        # hafta1_ornek_veri.csv'ye kaydedildi) ile rüzgarin geldigi yon
        # karsilastirilarak yayilma hizina yon etkisi ekleniyor.
        uyumlu = ruzgar_egime_uyumlu_mu(yangin["ruzgar_yonu_derece"], alan["egim_yonu"])
        hiz = yayilma_hizi_belirle(yangin["ruzgar_hizi"], alan["egim_derece"], uyumlu)

        sonuclar.append({
            "yangin_id": yangin["yangin_id"],
            "en_yakin_kritik_alan": alan["isim"],
            "kritik_alan_tipi": alan["tip"],
            "mesafe_metre": round(mesafe, 1),
            "egim_yonu": alan["egim_yonu"],
            "ruzgar_egime_uyumlu": uyumlu,
            "tahliye_skoru": tahliye,
            "yayilma_hizi": hiz,
        })
    return sonuclar


sonuc = yangin_kritik_alan_eslestir(mock_yangin_noktalari, kritik_alanlar)
sonuc_df = pd.DataFrame(sonuc)
sonuc_df

,yangin_id,en_yakin_kritik_alan,kritik_alan_tipi,mesafe_metre,egim_yonu,ruzgar_egime_uyumlu,tahliye_skoru,yayilma_hizi
0,yangin_01,İslamhaneleri,koy,470.4,kuzey,False,0.095,yavas
1,yangin_02,İkinci Bahar Huzur Evi,huzurevi,592.1,dogu,False,0.113,yavas


In [4]:
sonuc_df.to_csv("hafta2_eslesme_sonuc.csv", index=False, encoding="utf-8")
print("Kaydedildi: hafta2_eslesme_sonuc.csv")

Kaydedildi: hafta2_eslesme_sonuc.csv


## Notlar / sonraki adim
- `hesaplamalar.py` artik hem Hafta 1 hem Hafta 2 fonksiyonlarini iceriyor —
Esma bu moduluu oldugu gibi backend'e import edebilir.
- Yayilma hizi artik gercek egim yonu + ruzgar yonu karsilastirmasini
kullaniyor (onceki gecici `< 90` kestirimi kaldirildi).
- Mock yangin noktalari, Esma'nin `/yangin-noktalari` endpoint'i hazir olunca
gercek FIRMS verisiyle degistirilecek (fonksiyon imzasi ayni kaliyor).
- Hafta 3'te A* rota maliyetine `egim_cezasi()` eklenecek; bu moduldeki
`egim_derece` alani dogrudan orada kullanilabilir.